# Netflix Hit Proposal

Working notebook for the capstone deliverable: use real data from `netflix/data/*.csv` to (1) define what counts as a "hit", (2) find which pre-release-knowable features correlate with it, and (3) pitch an original series concept backed by that evidence.

This notebook runs on the **real** composite datasets (not the synthetic stand-in used in `netflix_review_concepts_capstone.ipynb`).

In [6]:
import numpy as np
import pandas as pd

netflix = pd.read_csv("netflix/data/netflix.titles.composite.csv")
netflix.shape

(691, 13)

## 1. Data completeness

Before defining anything, check how much of the composite dataset actually has each signal. `netflix_viewing_hours` and `netflix_weeks` come straight from Netflix's own Top 10 data and are complete for all 691 titles. The `tmdb_*` and `imdb_*` columns depend on a fuzzy-match join and are missing for a meaningful share of rows -- this is why they should stay out of the `hit` label itself (see Section 2).

In [7]:
missing_pct = netflix.isna().mean().mul(100).round(1).rename("missing_pct")
missing_pct.to_frame()

,missing_pct
key,0.0
netflix_viewing_hours,0.0
netflix_weeks,0.0
netflix_year_hint,0.0
netflix_title,0.0
netflix_clean_title,0.1
tmdb_title,43.0
tmdb_popularity,43.0
tmdb_vote_average,43.0
tmdb_vote_count,43.0


## 2. Do viewing hours, votes, and ratings actually move together?

Log1p-transform the heavily skewed count-like columns (`viewing_hours`, `tmdb_popularity`, `tmdb_vote_count`, `imdb_numVotes`) before correlating -- their raw distributions span orders of magnitude and would otherwise be dominated by a handful of outliers (e.g. Stranger Things).

**Finding:** `netflix_viewing_hours` correlates strongly with `netflix_weeks` (0.70) but only weakly with `imdb_averageRating` (0.18) and barely at all with `imdb_numVotes` (0.08). Reach (how much something got watched) and reception (how well it was rated) are largely independent signals here -- which is exactly why they shouldn't be collapsed into one `hit` label. `viewing_hours` and `weeks` are the only two that clearly measure the same underlying thing (Netflix-internal popularity), so those are the two combined into `hit_score` below.

In [8]:
log_cols = {
    "viewing_hours": np.log1p(netflix["netflix_viewing_hours"]),
    "weeks": np.log1p(netflix["netflix_weeks"]),
    "tmdb_popularity": np.log1p(netflix["tmdb_popularity"]),
    "tmdb_vote_average": netflix["tmdb_vote_average"],
    "tmdb_vote_count": np.log1p(netflix["tmdb_vote_count"]),
    "imdb_averageRating": netflix["imdb_averageRating"],
    "imdb_numVotes": np.log1p(netflix["imdb_numVotes"]),
}
corr_df = pd.DataFrame(log_cols)
corr_df.corr().round(2)

,viewing_hours,weeks,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,imdb_averageRating,imdb_numVotes
viewing_hours,1.00,0.81,0.39,0.27,0.44,0.18,0.08
weeks,0.81,1.00,0.27,0.13,0.25,0.14,0.07
tmdb_popularity,0.39,0.27,1.00,0.66,0.87,0.31,0.33
tmdb_vote_average,0.27,0.13,0.66,1.00,0.76,0.20,0.25
tmdb_vote_count,0.44,0.25,0.87,0.76,1.00,0.28,0.39
imdb_averageRating,0.18,0.14,0.31,0.20,0.28,1.00,0.02
imdb_numVotes,0.08,0.07,0.33,0.25,0.39,0.02,1.00


## 3. Defining `hit_score`

Decision: combine `netflix_viewing_hours` (total exposure) and `netflix_weeks` (durability -- did it stick around or spike and vanish) into one composite target. Ratings and vote counts stay out: they're weakly correlated with viewing hours (Section 2), would drop the usable sample from 691 to as few as 278 rows, and -- for the proposal use case -- don't exist yet for an unmade show anyway.

**Combination method:** z-score standardize each of `log1p(viewing_hours)` and `log1p(weeks)` independently, average them with equal weight, then min-max the result to `[0, 1]`.

Why equal weight and not a fitted weight: with exactly two already-standardized variables, PCA's first component is *always* the equal-weighted average, regardless of their correlation (the eigenvectors of `[[1, rho], [rho, 1]]` are always `(1,1)` and `(1,-1)`). There's also no ground-truth "hit" label available to fit a weight against via regression -- that would be circular, since `hit_score` is the thing being constructed. So equal weight isn't a shortcut; given two standardized inputs and no external criterion, it's what the data-driven approach reduces to.

In [9]:
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()

def minmax(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min())

z_hours = zscore(np.log1p(netflix["netflix_viewing_hours"]))
z_weeks = zscore(np.log1p(netflix["netflix_weeks"]))

netflix["hit_score"] = minmax((z_hours + z_weeks) / 2)
netflix[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score"]].sort_values(
    "hit_score", ascending=False
).head(10)

,netflix_title,netflix_viewing_hours,netflix_weeks,hit_score
396,Squid Game,2289500000,20,1.000000
103,Café con aroma de mujer,793740000,25,0.965821
661,Stranger Things,2874260000,13,0.937541
140,"Yo soy Betty, la fea",297560000,30,0.929910
549,Manifest,1446260000,16,0.926189
447,Money Heist,1170200000,14,0.886773
376,The Queen of Flow,561230000,16,0.858605
607,Ozark,751600000,13,0.841773
167,Bridgerton,1108800000,11,0.839614
598,Red Notice,453990000,14,0.819171


## 4. Sanity check

`hit_score` should reward both massive-but-brief spikes and modest-but-sustained runs, not just raw hours. Compare the ranking above to a plain `viewing_hours`-only ranking -- titles with high weeks but comparatively lower hours should move up; one-off spikes with very few weeks should move down relative to the hours-only ranking.

In [10]:
hours_rank = netflix["netflix_viewing_hours"].rank(ascending=False)
hit_rank = netflix["hit_score"].rank(ascending=False)
netflix["rank_shift"] = hours_rank - hit_rank  # positive = moved up under hit_score
netflix[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "hit_score", "rank_shift"]].sort_values(
    "rank_shift", ascending=False
).head(10)

,netflix_title,netflix_viewing_hours,netflix_weeks,hit_score,rank_shift
128,Beauty and the Beast,7480000,4,0.312790,263.5
377,Shiny_Flakes: The Teenage Drug Lord,10470000,4,0.336799,215.0
422,Grudge,13230000,5,0.388892,206.0
688,Xtreme,5390000,3,0.246083,206.0
180,Lost Bullet,6250000,3,0.256652,199.0
227,The Claus Family,21600000,6,0.453811,192.0
321,El infierno,13520000,4,0.355052,168.0
237,Confessions of an Invisible Girl,19730000,5,0.417426,166.0
569,Meenakshi Sundareshwar,9130000,3,0.283710,160.0
328,أصحاب ...ولا أعزّ,8730000,3,0.280512,156.0


## 5. Building the analysis dataset

Missing-value strategy: **complete-case analysis** -- keep only titles matched across all three sources (`netflix_title` always present, plus non-null `tmdb_title` and `imdb_title`). That's 391 of 691 rows.

**Assumption, stated explicitly:** we assume the upstream fuzzy-match pipeline's `tmdb_title`/`imdb_title` matches are correct. A quick spot-check did turn up some mismatches on numeric-heavy titles (e.g. `1917` matched to TMDB's `1916`) -- for a teaching exercise under a one-night deadline, fixing the matching pipeline itself is out of scope, so this is a known, documented limitation rather than something silently ignored.

Genre, language, and per-episode runtime come from `tmdb.titles.v3.csv`, joined on `tmdb_title == name`. Because some matched titles share a name with another, unrelated TMDB entry, ties are broken by keeping the highest-`vote_count` row for each name -- the more plausible candidate, though not guaranteed correct.

In [11]:
tmdb = pd.read_csv("netflix/data/tmdb.titles.v3.csv")

# complete-case: keep only titles matched in both tmdb and imdb
dataset = netflix.dropna(subset=["tmdb_title", "imdb_title"]).copy()

# de-dupe tmdb by name, keeping the highest-vote_count row for ambiguous names
tmdb_features = (
    tmdb.sort_values("vote_count", ascending=False)
    .drop_duplicates(subset="name", keep="first")[["name", "genres", "original_language", "episode_run_time"]]
)

dataset = dataset.merge(tmdb_features, left_on="tmdb_title", right_on="name", how="left")
dataset = dataset.drop(columns="name")

print(f"analysis dataset: {dataset.shape[0]} rows (from {len(netflix)} total, {len(netflix) - dataset.shape[0]} dropped for missing tmdb/imdb match)")
dataset[["netflix_title", "hit_score", "genres", "original_language", "episode_run_time"]].head(10)

analysis dataset: 391 rows (from 691 total, 300 dropped for missing tmdb/imdb match)


,netflix_title,hit_score,genres,original_language,episode_run_time
0,The 100,0.304167,"Sci-Fi & Fantasy, Drama, Action & Adventure",en,43
1,1000 Miles from Christmas,0.256272,Documentary,en,42
2,Love 101,0.172172,"Comedy, Drama",tr,45
3,Back to 15,0.197875,"Comedy, Drama",pt,40
4,1917,0.161430,Documentary,en,83
5,Formula 1: Drive to Survive,0.358672,Documentary,en,38
6,Blade Runner 2049,0.138388,Sci-Fi & Fantasy,en,0
7,211,0.138842,Documentary,es,0
8,21 Jump Street,0.166820,"Crime, Mystery, Drama",en,45
9,Monsters Inside: The 24 Faces of Billy Milligan,0.185331,"Documentary, Crime",en,60


## 6. Encoding features for regression

Three raw columns, three different encoding problems:

- **`genres`** is multi-label (a title can be `"Drama, Crime, Mystery"`), so it needs multi-hot encoding -- one binary column per genre, not a single categorical. Rare genres (`News`, `Western`, `Talk`, `War & Politics`, each under 10 titles) are dropped rather than encoded: with only 335 rows, a dummy that's `1` for two or three titles gives linear regression almost nothing to estimate a stable coefficient from. Titles left with none of the kept genres simply have all genre columns at `0` -- they fall into the same implicit baseline as the reference category dropped by `original_language`'s one-hot encoding below.
- **`original_language`** is a plain categorical, but has a long tail (24 languages, many appearing 1-3 times). Same reasoning as above: bucket anything under 10 titles into `other`, then one-hot with `drop_first=True` (English becomes the implicit baseline).
- **`episode_run_time`** has 55 rows (16%) sitting at exactly `0`, which isn't a plausible episode length -- almost certainly unpopulated TMDB data rather than a true zero. Treated as missing: replaced with the column median and flagged with a `runtime_missing` indicator, so the model can still tell those rows apart from titles with a genuinely short runtime.

In [12]:
# drop the rows where tmdb matched but had no genre data of its own
dataset = dataset.dropna(subset=["genres"]).reset_index(drop=True)

# --- genres: multi-hot, keep only genres with >= 10 occurrences ---
genre_lists = dataset["genres"].str.split(", ")
genre_counts = genre_lists.explode().str.strip().value_counts()
keep_genres = genre_counts[genre_counts >= 10].index.tolist()
for g in keep_genres:
    dataset[f"genre_{g}"] = genre_lists.apply(lambda lst: g in [x.strip() for x in lst]).astype(int)

# --- original_language: bucket rare languages, then one-hot ---
lang_counts = dataset["original_language"].value_counts()
keep_langs = lang_counts[lang_counts >= 10].index.tolist()
dataset["language_bucketed"] = dataset["original_language"].where(
    dataset["original_language"].isin(keep_langs), "other"
)
lang_dummies = pd.get_dummies(dataset["language_bucketed"], prefix="lang", drop_first=True).astype(int)
dataset = pd.concat([dataset, lang_dummies], axis=1)

# --- episode_run_time: treat 0 as missing, impute with median, flag it ---
dataset["runtime_missing"] = (dataset["episode_run_time"] == 0).astype(int)
runtime_clean = dataset["episode_run_time"].replace(0, np.nan)
dataset["episode_run_time_imputed"] = runtime_clean.fillna(runtime_clean.median())

feature_cols = (
    [f"genre_{g}" for g in keep_genres]
    + list(lang_dummies.columns)
    + ["episode_run_time_imputed", "runtime_missing"]
)
print(f"final dataset: {dataset.shape[0]} rows, {len(feature_cols)} features")
dataset[["netflix_title", "hit_score", *feature_cols]].head(5)

final dataset: 335 rows, 20 features


,netflix_title,hit_score,genre_Drama,genre_Crime,genre_Comedy,genre_Sci-Fi & Fantasy,genre_Action & Adventure,genre_Mystery,genre_Documentary,genre_Animation,...,genre_Soap,genre_Kids,lang_es,lang_fr,lang_ja,lang_ko,lang_other,lang_pt,episode_run_time_imputed,runtime_missing
0,The 100,0.304167,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,43.0,0
1,1000 Miles from Christmas,0.256272,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,42.0,0
2,Love 101,0.172172,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,45.0,0
3,Back to 15,0.197875,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,40.0,0
4,1917,0.161430,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,83.0,0


## 7. Correlation check before fitting

Two things worth checking before regression: (1) does each feature actually relate to `hit_score` at all, and (2) are any features so correlated with each other that the regression coefficients will be unstable (multicollinearity)?

In [13]:
target_corr = (
    dataset[feature_cols + ["hit_score"]].corr()["hit_score"].drop("hit_score").sort_values(key=abs, ascending=False)
)
print("--- correlation with hit_score ---")
print(target_corr.round(3))

fcorr = dataset[feature_cols].corr()
high_pairs = [
    (a, b, round(fcorr.loc[a, b], 3))
    for i, a in enumerate(feature_cols)
    for b in feature_cols[i + 1 :]
    if abs(fcorr.loc[a, b]) > 0.5
]
print()
print("--- feature pairs with |r| > 0.5 (multicollinearity risk) ---")
print(high_pairs if high_pairs else "none")

--- correlation with hit_score ---
genre_Drama                 0.294
lang_ko                     0.215
lang_other                 -0.156
genre_Documentary          -0.128
genre_Mystery               0.111
genre_Reality              -0.101
lang_fr                    -0.096
genre_Animation            -0.090
episode_run_time_imputed    0.085
genre_Family               -0.077
runtime_missing            -0.071
lang_es                     0.067
genre_Kids                 -0.059
lang_ja                    -0.057
genre_Crime                 0.054
lang_pt                    -0.048
genre_Soap                  0.045
genre_Action & Adventure    0.034
genre_Sci-Fi & Fantasy      0.011
genre_Comedy               -0.009
Name: hit_score, dtype: float64

--- feature pairs with |r| > 0.5 (multicollinearity risk) ---
none


**Findings:** no feature pair exceeds `|r| > 0.5`, so multicollinearity isn't a concern for the regression. Correlation with `hit_score` is weak across the board -- unsurprising, since genre/language/runtime alone are a coarse description of a show -- but `genre_Drama` (0.29) and `lang_ko` (0.22, Korean-language titles) stand out as the two features most worth watching in the regression coefficients.

## 8. Fitting the regression

Same normal-equations approach as Module 2 (`beta = (X'X)^-1 X'y`), extended with standard errors, t-stats, and p-values so coefficients can be judged for statistical significance, not just sign and magnitude -- `numpy` + `scipy.stats.t`, no new library needed.

In [14]:
from scipy import stats

X_raw = dataset[feature_cols].to_numpy(dtype=float)
y = dataset["hit_score"].to_numpy(dtype=float)
n, k = X_raw.shape
X = np.column_stack([np.ones(n), X_raw])
p = X.shape[1]

XtX_inv = np.linalg.inv(X.T @ X)
beta = XtX_inv @ X.T @ y
resid = y - X @ beta
rss = np.sum(resid ** 2)
sigma2 = rss / (n - p)
se = np.sqrt(np.diag(sigma2 * XtX_inv))
t_stats = beta / se
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - p))

tss = np.sum((y - y.mean()) ** 2)
r2 = 1 - rss / tss
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p)

regression_result = pd.DataFrame(
    {"coef": beta, "se": se, "t": t_stats, "p_value": p_values},
    index=["intercept", *feature_cols],
).sort_values("p_value")

print(f"n={n}, R2={r2:.4f}, adj_R2={adj_r2:.4f}")
regression_result.round(4)

n=335, R2=0.1657, adj_R2=0.1126


,coef,se,t,p_value
intercept,0.3083,0.0473,6.5161,0.0000
lang_other,-0.1162,0.0326,-3.5631,0.0004
genre_Drama,0.1101,0.0335,3.2815,0.0011
lang_fr,-0.1128,0.0595,-1.8968,0.0588
lang_ko,0.0708,0.0397,1.7866,0.0750
lang_pt,-0.0709,0.0619,-1.1448,0.2532
lang_ja,-0.0548,0.0496,-1.1038,0.2705
genre_Family,-0.0408,0.0503,-0.8109,0.4180
genre_Soap,0.0413,0.0610,0.6769,0.4990
genre_Mystery,0.0196,0.0311,0.6313,0.5283


**Results:** R² = 0.166, adjusted R² = 0.113 -- low, as expected, since `genre`/`original_language`/`episode_run_time` are a coarse sketch of a show, not a full description of its writing or marketing. Still, two coefficients clear p < 0.05:

- **`genre_Drama`: +0.110 (p = 0.001)** -- holding language and runtime fixed, being tagged Drama is associated with a meaningfully higher `hit_score`. Consistent with `genre_Drama`'s 0.29 correlation from Section 7.
- **`lang_other` (niche/long-tail language): -0.116 (p < 0.001)** -- titles in a language outside the six most common (en/es/ko/ja/fr/pt) score meaningfully lower, relative to the English baseline.

Two more are borderline (0.05 < p < 0.10) and point in *opposite* directions -- worth naming as directional evidence in the proposal, but neither is strong enough to lean on alone: `lang_ko` is **+0.071** (p = 0.075, Korean-language titles trending higher), while `lang_fr` is **-0.113** (p = 0.059, French-language titles trending lower). Everything else (most genres, `episode_run_time`, `runtime_missing`) is statistically indistinguishable from zero at this sample size -- absence of evidence, not evidence that they don't matter; a coarse 335-row, pre-release-only feature set simply can't resolve a subtler effect than that.

## Next steps (not yet done)

- Cast/crew historical track record (aggregate the *past* performance of attached writers/directors via the IMDb dataset) -- deferred for now.
- Pick a Polti dramatic situation for the pitched show and write the proposal, using `genre_Drama` (+), `lang_other` (-), and the two borderline language coefficients as supporting evidence.